Validation Set (human validation)

In [4]:
#load csv
import os
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../validation/interrater_samples_complete.csv')
# keep only the folllowing index,publication,year,authors,cites,title,abstract, revised_classification,revised_confidence,revised_reason ,interrater_1_classification,interrater_1_confidence,interrater_1_reason,interrater_2_classification,interrater_2_confidence,interrater_2_reason
df = df[['index','publication','year','authors','cites','title','abstract',
         'revised_classification','revised_confidence','revised_reason',
         'interrater_1_classification','interrater_1_confidence','interrater_1_reason',
         'interrater_2_classification','interrater_2_confidence','interrater_2_reason']]
# keep if interrater_1_classification euqals interrater_2_classification
df_groundtruth = df[df['interrater_1_classification'] == df['interrater_2_classification']]


# --- Count samples per class ---
print("Number of samples per class in ground truth:")
for cls, count in df_groundtruth['interrater_1_classification'].value_counts().items():
    print(f"{cls}: {count}")

# --- Prepare for stratified split ---
y = df_groundtruth['interrater_1_classification']

# Step 1: Train + Valid (80%) / Test (20%)
train_val, test = train_test_split(
    df_groundtruth,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Step 2: Train (60%) / Validation (20%)
train, val = train_test_split(
    train_val,
    test_size=0.25,   # 0.25 of 0.8 = 0.2 total for validation
    stratify=train_val['interrater_1_classification'],
    random_state=42
)

# --- Check proportions ---
def show_distribution(df, name):
    dist = df['interrater_1_classification'].value_counts(normalize=True) * 100
    print(f"\n{name} set class distribution (%):")
    print(dist.round(2))

show_distribution(train, "Train")
show_distribution(val, "Validation")
show_distribution(test, "Test")



train.to_csv('../validation/train_groundtruth.csv', index=False)
val.to_csv('../validation/val_groundtruth.csv', index=False)
test.to_csv('../validation/test_groundtruth.csv', index=False)

Number of samples per class in ground truth:
Unrelated: 27
Supports CLD: 27
Supports PTLDS: 15
Animal Study: 14
Neutral: 8

Train set class distribution (%):
interrater_1_classification
Unrelated         31.48
Supports CLD      29.63
Supports PTLDS    16.67
Animal Study      14.81
Neutral            7.41
Name: proportion, dtype: float64

Validation set class distribution (%):
interrater_1_classification
Supports CLD      27.78
Unrelated         27.78
Supports PTLDS    16.67
Animal Study      16.67
Neutral           11.11
Name: proportion, dtype: float64

Test set class distribution (%):
interrater_1_classification
Supports CLD      31.58
Unrelated         26.32
Animal Study      15.79
Supports PTLDS    15.79
Neutral           10.53
Name: proportion, dtype: float64


Sample for 250

In [ ]:
df_relevant = pd.read_csv("../datasets/classification_results_feb_2025.csv")
df_all = pd.read_csv("../datasets/classification_combined_df-8k-dataset-minus2025-abstracts.csv")



df_new = []
for _, row in df_all.iterrows():
    if row["abstract"] in df_relevant["abstract"].values:
        classification = df_relevant[df_relevant["abstract"] == row["abstract"]]["revised_classification"].values[0]
    else:
        # print(row["classification"])
        classification = "Unrelated" if row["classification"] == "Definitely Unrelated" else "Animal Study"
    df_new.append(
        {
            "index": row["index"],
            "publication": row["publication"],
            "title": row["title"],
            "authors": row["authors"],
            "year": row["year"],
            "cites": row["cites"],
            "abstract": row["abstract"],
            "classification":  classification,
            
        }
    )
df_final = pd.DataFrame(df_new)
df_final.to_csv("../datasets/classification_all_8k.csv", index=False)


df_final = pd.read_csv("../datasets/classification_all_8k.csv")
# sample df final 300 samples with balanced classes
df_sample = pd.concat(
    [
        df_final[df_final["classification"] == "Supports PTLDS"].sample(50, random_state=42),
        df_final[df_final["classification"] == "Supports CLD"].sample(50, random_state=42),
        df_final[df_final["classification"] == "Animal Study"].sample(50, random_state=42),
        df_final[df_final["classification"] == "Unrelated"].sample(50, random_state=42),
        df_final[df_final["classification"] == "Neutral"].sample(50, random_state=42),
    ],
    ignore_index=True,
)
df_sample.to_csv("../datasets/classification_300_samples_balanced.csv", index=False)